# Notebook 06-CIC — Cross-Model SHAP Agreement on CIC-IDS2017

**Project:** Calibrated and Stability-Aware Explainable Intrusion Detection
**Author:** Md Anas Biswas, University of Portsmouth
**Dataset:** CIC-IDS2017

## Purpose

Apply Krishna et al. 2022 six agreement metrics to CIC-IDS2017 SHAP rankings. Replicates the NSL-KDD analysis from Notebook 06 to demonstrate that the cross-model agreement findings hold across datasets.

## Key paper claim being tested

On NSL-KDD: tree models agree strongly (rank correlation +0.47 to +0.55), tree-DNN pairs disagree (-0.30 to +0.06). Does this pattern replicate on CIC-IDS2017?

## Method

Reuses CIC-IDS2017 SHAP arrays from Notebook 04-CIC. Aligns RF subsample (715 samples) with other models' SHAP (sliced to same indices). Computes 6 metrics at k ∈ {5, 10, 15}, aggregate + per-class for 5-class models.

**Runs on CPU.** No GPU needed.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
REPO = '/content/drive/MyDrive/XIDS_Research/xids-research'
os.chdir(REPO)

for f in ['.gitconfig', '.git-credentials']:
    src = f'/content/drive/MyDrive/XIDS_Research/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'/root/{f}')
        if f == '.git-credentials':
            os.chmod(f'/root/{f}', 0o600)

!git pull
print(f'✓ Ready in: {os.getcwd()}')

Mounted at /content/drive
Already up to date.
✓ Ready in: /content/drive/MyDrive/XIDS_Research/xids-research


In [2]:
import numpy as np
import pandas as pd
import json
from pathlib import Path
from scipy.stats import spearmanr
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
np.random.seed(SEED)

PROCESSED = Path(REPO) / 'data' / 'processed' / 'cic_ids2017'
CALIB_DIR = Path(REPO) / 'calibrators' / 'cic_ids2017'
SHAP_DIR = Path(REPO) / 'shap_values' / 'cic_ids2017'
FIG_DIR = Path(REPO) / 'results' / 'figures'
TABLES_DIR = Path(REPO) / 'results' / 'tables'
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

y_test_b = np.load(PROCESSED / 'y_test_binary.npy')
y_test_5 = np.load(PROCESSED / 'y_test_5class.npy')
idx_eval = np.load(CALIB_DIR / 'idx_eval.npy')
rf_subsample_idx = np.load(SHAP_DIR / 'rf_subsample_idx.npy')

y_eval_b = y_test_b[idx_eval]
y_eval_5 = y_test_5[idx_eval]
y_sub_b = y_eval_b[rf_subsample_idx]
y_sub_5 = y_eval_5[rf_subsample_idx]

with open(PROCESSED / 'feature_names.json') as f:
    FEATURE_NAMES = json.load(f)
with open(PROCESSED / 'class_mappings.json') as f:
    class_info = json.load(f)
INT_TO_CATEGORY = {int(k): v for k, v in class_info['multiclass_5'].items()}
CLASS_NAMES_5 = [INT_TO_CATEGORY[i] for i in range(5)]

print(f'RF subsample: {len(rf_subsample_idx)}')
print(f'5-class distribution in RF subsample: {np.bincount(y_sub_5)}')

RF subsample: 715
5-class distribution in RF subsample: [200 200 200 112   3]


---
## Step 1 — Load and align SHAP arrays

Reuses CIC-IDS2017 SHAP from Notebook 04-CIC. RF was on 715 samples; others on full eval (20,004). Slice others to RF's indices.

In [ ]:
MODELS_BINARY = ['rf_binary_cw', 'xgb_binary_cw', 'dnn_binary_cw']
MODELS_5CLASS = ['rf_5class_smote', 'xgb_5class_smote', 'dnn_5class_smote']
PAIRS = [('rf', 'xgb'), ('rf', 'dnn'), ('xgb', 'dnn')]

def load_and_align(name):
    arr = np.load(SHAP_DIR / f'{name}_shap.npy')
    if name.startswith('rf'):
        return arr  # already 715 samples
    return arr[rf_subsample_idx]

SHAP_BINARY = {arch.split('_')[0]: load_and_align(arch) for arch in MODELS_BINARY}
SHAP_5CLASS = {arch.split('_')[0]: load_and_align(arch) for arch in MODELS_5CLASS}

print('Aligned SHAP shapes:')
print('  Binary:')
for arch, sv in SHAP_BINARY.items():
    print(f'    {arch}: {sv.shape}')
print('  5-class:')
for arch, sv in SHAP_5CLASS.items():
    print(f'    {arch}: {sv.shape}')

---
## Step 2 — The six Krishna metrics (same as NSL-KDD)

In [ ]:
def feature_agreement(shap_a, shap_b, k):
    top_a = set(np.argsort(-np.abs(shap_a))[:k])
    top_b = set(np.argsort(-np.abs(shap_b))[:k])
    return len(top_a & top_b) / k

def rank_agreement(shap_a, shap_b, k):
    rank_a = np.argsort(-np.abs(shap_a))[:k]
    rank_b = np.argsort(-np.abs(shap_b))[:k]
    return sum(1 for i in range(k) if rank_a[i] == rank_b[i]) / k

def sign_agreement(shap_a, shap_b, k):
    top_a = set(np.argsort(-np.abs(shap_a))[:k])
    top_b = set(np.argsort(-np.abs(shap_b))[:k])
    common = top_a & top_b
    if not common: return 0.0
    return sum(1 for f in common if np.sign(shap_a[f]) == np.sign(shap_b[f])) / len(common)

def signed_rank_agreement(shap_a, shap_b, k):
    rank_a = np.argsort(-np.abs(shap_a))[:k]
    rank_b = np.argsort(-np.abs(shap_b))[:k]
    matches = sum(1 for i in range(k)
                  if rank_a[i] == rank_b[i] and np.sign(shap_a[rank_a[i]]) == np.sign(shap_b[rank_a[i]]))
    return matches / k

def rank_correlation(shap_a, shap_b, k):
    top_a_idx = np.argsort(-np.abs(shap_a))[:k]
    ranks_a = np.argsort(np.argsort(-np.abs(shap_a)[top_a_idx])) + 1
    ranks_b = np.argsort(np.argsort(-np.abs(shap_b)[top_a_idx])) + 1
    if k < 2: return 0.0
    corr, _ = spearmanr(ranks_a, ranks_b)
    return float(corr) if not np.isnan(corr) else 0.0

def pairwise_rank_agreement(shap_a, shap_b, k):
    top_a = np.argsort(-np.abs(shap_a))[:k]
    abs_b = np.abs(shap_b)
    n_pairs = 0; agree = 0
    for i in range(k):
        for j in range(i+1, k):
            fi, fj = top_a[i], top_a[j]
            n_pairs += 1
            if abs_b[fi] >= abs_b[fj]:
                agree += 1
    return agree / n_pairs if n_pairs > 0 else 0.0

METRICS = {
    'feature_agreement': feature_agreement,
    'rank_agreement': rank_agreement,
    'sign_agreement': sign_agreement,
    'signed_rank_agreement': signed_rank_agreement,
    'rank_correlation': rank_correlation,
    'pairwise_rank_agreement': pairwise_rank_agreement,
}
print(f'✓ {len(METRICS)} metric functions defined')

---
## Step 3 — Binary results

In [ ]:
K_VALUES = [5, 10, 15]

def get_binary_vec(arr, i):
    return arr[i, :, 1] if arr.ndim == 3 else arr[i]

def compute_pair_metrics_binary(arr_a, arr_b, k):
    n = arr_a.shape[0]
    out = {m: 0.0 for m in METRICS}
    for i in range(n):
        va = get_binary_vec(arr_a, i)
        vb = get_binary_vec(arr_b, i)
        for m, fn in METRICS.items():
            out[m] += fn(va, vb, k)
    return {m: out[m] / n for m in METRICS}

print('Binary — CIC-IDS2017:\n')
binary_results = []
for (arch_a, arch_b) in PAIRS:
    for k in K_VALUES:
        m = compute_pair_metrics_binary(SHAP_BINARY[arch_a], SHAP_BINARY[arch_b], k)
        row = {'Pair': f'{arch_a}-{arch_b}', 'k': k, 'Class': 'all', 'Target': 'binary'}
        row.update(m)
        binary_results.append(row)
        print(f'  {arch_a}-{arch_b} @ k={k}: ' + ', '.join(f'{mn[:8]}={v:+.3f}' for mn, v in m.items()))

df_binary = pd.DataFrame(binary_results)

---
## Step 4 — 5-class results (aggregate + per-class)

In [ ]:
def get_agg_vec(arr, i): return np.abs(arr[i]).sum(axis=-1)
def get_class_vec(arr, i, c): return arr[i, :, c]

def compute_pair_5class(arr_a, arr_b, k, sample_idx, mode, c=None):
    if len(sample_idx) == 0:
        return {m: np.nan for m in METRICS}
    out = {m: 0.0 for m in METRICS}
    for i in sample_idx:
        if mode == 'aggregate':
            va = get_agg_vec(arr_a, i); vb = get_agg_vec(arr_b, i)
        else:
            va = get_class_vec(arr_a, i, c); vb = get_class_vec(arr_b, i, c)
        for m, fn in METRICS.items():
            out[m] += fn(va, vb, k)
    return {m: out[m] / len(sample_idx) for m in METRICS}

print('5-class aggregate — CIC-IDS2017:\n')
results_5class_agg = []
all_idx = np.arange(SHAP_5CLASS['rf'].shape[0])
for (arch_a, arch_b) in PAIRS:
    for k in K_VALUES:
        m = compute_pair_5class(SHAP_5CLASS[arch_a], SHAP_5CLASS[arch_b], k, all_idx, 'aggregate')
        row = {'Pair': f'{arch_a}-{arch_b}', 'k': k, 'Class': 'aggregate', 'Target': '5class'}
        row.update(m)
        results_5class_agg.append(row)
        print(f'  {arch_a}-{arch_b} @ k={k}: ' + ', '.join(f'{mn[:8]}={v:+.3f}' for mn, v in m.items()))

df_5class_agg = pd.DataFrame(results_5class_agg)

In [ ]:
print('5-class per-class — CIC-IDS2017:\n')
results_5class_pc = []
for c in range(5):
    class_idx = np.where(y_sub_5 == c)[0]
    if len(class_idx) < 5:
        print(f'  Skipping {CLASS_NAMES_5[c]}: only {len(class_idx)} samples')
        continue
    for (arch_a, arch_b) in PAIRS:
        for k in K_VALUES:
            m = compute_pair_5class(SHAP_5CLASS[arch_a], SHAP_5CLASS[arch_b], k, class_idx, 'perclass', c=c)
            row = {'Pair': f'{arch_a}-{arch_b}', 'k': k, 'Class': CLASS_NAMES_5[c], 'Target': '5class'}
            row.update(m)
            results_5class_pc.append(row)
    print(f'  ✓ {CLASS_NAMES_5[c]} (n={len(class_idx)})')

df_5class_pc = pd.DataFrame(results_5class_pc)

---
## Step 5 — Save + headline output

In [ ]:
all_results = pd.concat([df_binary, df_5class_agg, df_5class_pc], ignore_index=True)
all_results.to_csv(TABLES_DIR / 'cic_krishna_full.csv', index=False)

agg = pd.concat([df_binary, df_5class_agg], ignore_index=True)
agg.to_csv(TABLES_DIR / 'cic_krishna_aggregate.csv', index=False)
df_5class_pc.to_csv(TABLES_DIR / 'cic_krishna_perclass.csv', index=False)

headline = agg[agg['k'] == 10][['Target', 'Pair'] + list(METRICS.keys())]
headline.to_csv(TABLES_DIR / 'cic_krishna_summary.csv', index=False)

print('=== AGGREGATE RESULTS (k=10, CIC-IDS2017) ===')
print(headline.to_string(index=False, float_format='%.3f'))
print()
print('=== PER-CLASS (k=10, CIC-IDS2017) ===')
pc10 = df_5class_pc[df_5class_pc['k'] == 10][['Class', 'Pair'] + list(METRICS.keys())]
print(pc10.to_string(index=False, float_format='%.3f'))

---
## Step 6 — Heatmap

In [ ]:
for target in ['binary', '5class']:
    df = agg[(agg['Target'] == target) & (agg['k'] == 10)].copy()
    pivot = df.set_index('Pair')[list(METRICS.keys())]
    fig, ax = plt.subplots(figsize=(11, 4))
    sns.heatmap(pivot, annot=True, fmt='.2f', cmap='RdYlGn', vmin=-1, vmax=1,
                cbar_kws={'label': 'Agreement'}, ax=ax)
    ax.set_title(f'Krishna agreement — CIC-IDS2017 {target} (k=10)')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.savefig(FIG_DIR / f'cic_krishna_heatmap_{target}.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Commit
os.chdir(REPO)
!git add notebooks/06_cic_shap_agreement.ipynb
!git add results/
!git status --short
!git commit -m 'Notebook 06-CIC: Krishna cross-model agreement on CIC-IDS2017'
!git push

## Summary

Cross-model agreement metrics computed on CIC-IDS2017. To compare to NSL-KDD findings:

**Expected (replication of NSL-KDD pattern):**
- RF ↔ XGB: high rank correlation (~+0.5)
- RF ↔ DNN: low or negative rank correlation
- XGB ↔ DNN: low or negative rank correlation

If this pattern replicates, **the tree-vs-DNN architectural divergence becomes a cross-dataset finding** for the paper.
